In [0]:
silver_df = spark.table("realtimedatabrickhybrid.realtime_pipeline_silver.online_retail")

silver_df.display()

In [0]:
from pyspark.sql.functions import sum

total_revenue_df = silver_df.agg(
    sum("TotalAmount").alias("total_revenue")
)

total_revenue_df.display()

In [0]:
revenue_by_country_df = (
    silver_df
    .groupBy("Country")
    .agg(sum("TotalAmount").alias("total_revenue"))
)

revenue_by_country_df.display()

In [0]:
from pyspark.sql.functions import desc

top_customers_df = (
    silver_df
    .filter("CustomerID IS NOT NULL")
    .groupBy("CustomerID")
    .agg(sum("TotalAmount").alias("total_spent"))
    .orderBy(desc("total_spent"))
)
top_customers_df.display()

In [0]:
from pyspark.sql.functions import to_date

daily_sales_df = (
    silver_df
    .withColumn("date", to_date("InvoiceDate"))
    .groupBy("date")
    .agg(sum("TotalAmount").alias("daily_revenue"))
)
daily_sales_df.display()

In [0]:
total_revenue_df.write.mode("overwrite").saveAsTable("realtimedatabrickhybrid.realtime_pipeline_gold.total_revenue")

revenue_by_country_df.write.mode("overwrite").saveAsTable("realtimedatabrickhybrid.realtime_pipeline_gold.revenue_by_country")

top_customers_df.write.mode("overwrite").saveAsTable("realtimedatabrickhybrid.realtime_pipeline_gold.top_customers")

daily_sales_df.write.mode("overwrite").saveAsTable("realtimedatabrickhybrid.realtime_pipeline_gold.daily_sales")

In [0]:
%run ../common/01_config

In [0]:
silver_df_adls = spark.table("realtimedatabrickhybrid.realtime_pipeline_silver.online_retail")
silver_df.display()

In [0]:
from pyspark.sql.functions import sum, desc, to_date

total_revenue_df = silver_df.agg(sum("TotalAmount").alias("total_revenue"))

revenue_by_country_df = (
    silver_df.groupBy("Country")
    .agg(sum("TotalAmount").alias("total_revenue"))
)

top_customers_df = (
    silver_df.filter("CustomerID IS NOT NULL")
    .groupBy("CustomerID")
    .agg(sum("TotalAmount").alias("total_spent"))
    .orderBy(desc("total_spent"))
)

daily_sales_df = (
    silver_df.withColumn("sales_date", to_date("InvoiceDate"))
    .groupBy("sales_date")
    .agg(sum("TotalAmount").alias("daily_revenue"))
)

total_revenue_df.display()

In [0]:
%run ../common/02_adls_connection

In [0]:
total_revenue_df.write.format("delta").mode("overwrite").save(gold_total_revenue)
revenue_by_country_df.write.format("delta").mode("overwrite").save(gold_revenue_by_country)
top_customers_df.write.format("delta").mode("overwrite").save(gold_top_customers)
daily_sales_df.write.format("delta").mode("overwrite").save(gold_daily_sales)